In [12]:
import pandas as pd
from IPython.display import clear_output

PATH = "../eval/label_set_relabel.csv"   # for the test set use ../eval/label_set_test.csv
df = pd.read_csv(PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig")
CODES = {"l": "Low", "m": "Medium", "h": "High", "n": "NA"}


def save():
    df.to_csv(PATH, index=False, encoding="utf-8-sig")


for i, row in df.iterrows():
    if row["risk"]:
        continue
    clear_output(wait=True)
    print(f"Item {row['id']} / {len(df)}    Clause: {row['clause']}")
    print("Parties:", row["parties"])
    print("=" * 80)
    import textwrap
    for line in row["text"].split("\n"):
        print(textwrap.fill(line, 100))
    print("=" * 80)
    while True:
        raw = input("l m h n (add u if unsure, e.g. mu) or q: ").strip().lower()
        code, unsure = raw[:1], raw[1:] == "u"
        if code in CODES or code == "q":
            break
    if code == "q":
        break
    df.at[i, "risk"] = CODES[code]
    df.at[i, "unsure"] = "1" if unsure else ""
    df.at[i, "customer"] = input("Customer (Enter = unclear): ").strip() or "unclear"
    df.at[i, "reason"] = input("Reason (Enter = skip): ").strip()
    save()

print("labeled:", (df["risk"] != "").sum(), "of", len(df))

Item 42 / 15    Clause: Uncapped Liability
Parties: the "Company" or "Phreesia" | Company | Phreesia, Inc. | Allscripts Healthcare, LLC | Allscripts | Phreesia
s arising from a Party's gross negligence or more culpable conduct, including any willful misconduct
or intentionally wrongful acts; (e) losses for death, bodily injury, or damage to real or tangible
personal property arising out of or relating to a Party's negligent or more culpable acts or
omissions; or (f) a Party's obligation to pay attorneys' fees and other costs pursuant to Section
28.9(e). >>> In addition, the limitations in Section 23.1(b) will not apply (1) to Company's
indemnification obligations under Section 22.1(a) or (2) Allscripts indemnification obligations
under Section 22.3(a), unless the Company's or Allscripts' indemnification obligation under Section
22.1(a) or 22.3(a), as the case may be, relates to the losses and obligations described in
subclauses (a) through (f) of the preceding sentence. [***]. <<<

23.

In [8]:
import pandas as pd
orig = pd.read_csv("../eval/label_set.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
s = orig[orig["risk"].isin(["Low", "Medium", "High"])].sample(15, random_state=1).copy()
for col in ["customer", "risk", "reason", "unsure"]:
    s[col] = ""
s.to_csv("../eval/label_set_relabel.csv", index=False, encoding="utf-8-sig")

In [11]:
re = pd.read_csv("../eval/label_set_relabel.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
first = orig.set_index("id").loc[re["id"], "risk"].values
print("same label both times:", (first == re["risk"].values).mean())

same label both times: 0.0


In [10]:
import os
import pandas as pd

RELABEL = "../eval/label_set_relabel.csv"
orig = pd.read_csv("../eval/label_set.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")

if os.path.exists(RELABEL):
    print("relabel file already exists, not overwriting")
else:
    s = orig[orig["risk"].isin(["Low", "Medium", "High"])].sample(15, random_state=1).copy()
    for col in ["customer", "risk", "reason", "unsure"]:
        s[col] = ""
    s.to_csv(RELABEL, index=False, encoding="utf-8-sig")
    print("created", RELABEL)

relabel file already exists, not overwriting
